# Lecture 19: Introduction to Partial Differential Equation Solvers, and Modular Coding

We will now look at partial differential equations.  There are [3
broad classes of
PDEs](https://en.wikiversity.org/wiki/Partial_differential_equations#Elliptic,_Hyperbolic,_and_Parabolic_PDEs)
that we will study: hyperbolic, elliptic, and parabolic PDEs.  These
will each require different solution methods. We will start with a brief overview of PDE classes and the types of problems these are most applicable to.

To introduce these PDE methods, the code we will need to utenlize will begin to be sufficiently complicated that additional tools to simplify these calculations will need to be introduced. Here, we introduce the concept of *modular coding*, or loading scripts with pre-loaded python functions and classes that can be used in your Jupyter Notebook cells.

We will start by introducing methods to solve hyperbolic PDEs.

For more in-depth discussions of the methods used for PDEs in astrophysics,
you can read the notes here: [_Introduction to Computational Astrophysical Hydrodynamics_](https://open-astrophysics-bookshelf.github.io/numerical_exercises/).

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-19/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We have used material from the following sources: \
https://zingale.github.io/computational_astrophysics/advection/pdes.html \
https://www.w3schools.com/python/python_modules.asp \
https://sbu-python-class.github.io/python-science/09-packages/python-modules.html

We also thank Shane Davis for material that went into the design of this lecture.

In [ ]:
# Nice things to have this lecture
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## Hyperbolic PDEs

The canonical hyperbolic PDE is the wave equation:

$$\frac{\partial^2 \phi}{\partial t^2} = c^2 \frac{\partial^2 \phi}{\partial x^2}$$

The solution to this is traveling waves moving at speed $c$ in both directions:

$$\phi(x, t) = A f_0(x - ct) + B g_0(x + ct)$$

A simpler example of this that we will look at is the _linear advection equation_:

$$\frac{\partial a}{\partial t} + u \frac{\partial a}{\partial x} = 0$$

The defining feature of hyperbolic PDEs is that there is a finite, real speed at which the solution changes.

In astrophysics, advection-like equations describe fluid dynamics.

## Elliptic PDEs

The Poisson equation is the canonical elliptic PDE:

$$\nabla^2 \phi = f$$

Notice that there is no time variable here.  Information about the solution
moves at infinite speed, and the solution depends only on the boundary conditions and the source, $f$.

In astrophysics, the Poisson equation is used to obtain the gravitational
potential, via [_Poisson's equation_](https://en.wikipedia.org/wiki/Poisson%27s_equation#Newtonian_gravity)

$$\nabla^2 \Phi = 4 \pi G \rho$$

## Parabolic PDEs

The heat equation is the canonical parabolic PDE:

$$\frac{\partial \phi}{\partial t} = k \frac{\partial^2 \phi}{\partial x^2}$$

As with elliptic PDEs, information about the solution is communicated instantaneously&mdash;there is no finite propagate speed like the hyperbolic case.
But this is time-dependent.  

Diffusion of photons or heat in stars is perhaps the main area where parabolic PDEs
arise in astrophysics.

# Advection

In astrophysics, many systems (stars, the ISM, gas in galaxies) is modeled as
a fluid, following the equations of hydrodynamics:

\begin{align*}
\frac{\partial \rho}{\partial t} + \nabla \cdot (\rho {\bf U}) &= 0 \\
\frac{\partial (\rho {\bf U})}{\partial t} + \nabla \cdot (\rho {\bf U}{\bf U}) + \nabla p 
&= 0 \\
\frac{\partial (\rho E)}{\partial t} + \nabla \cdot (\rho {\bf U} E + {\bf U}p) &= 0
\end{align*}

These express conservation of mass, momentum, and energy.  If we expand out the diveragence
in the first equation, we get:

$$\frac{\partial \rho}{\partial t} + {\bf U} \cdot \nabla \rho + \rho \nabla \cdot {\bf U} = 0 $$

This has the form of an advection equation.  

**Note:**
Our model equation for this, in one-dimension
will be the _linear advection equation_:

$$\frac{\partial a}{\partial t} + u \frac{\partial a}{\partial x} = 0$$

Looking at the second equation (momentum), and expanding out and simplifying, we have:

$$\frac{\partial {\bf U}}{\partial t} + {\bf U} \cdot \nabla {\bf U} + \frac{1}{\rho} \nabla p = 0$$

This is a nonlinear equation (because of the ${\bf U}\cdot \nabla {\bf
U}$ term).  This nonlinearity admits a rich diversity of solutions:
rarefactions, shocks, and turbulence.

**Note:**
Our model equation for this will be the [inviscid Burgers'
equation](https://en.wikipedia.org/wiki/Burgers%27_equation):

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = 0$$

# Linear Advection Equation

The equations of hydrodynamics are a system of nonlinear partial differential equations that involve advection of mass, momentum, and energy.  To get a feel for the solution of these equations, we will start with the simple [linear advection equation](https://en.wikipedia.org/wiki/Advection#The_advection_equation).

$$a_t + u a_x = 0$$

This requires initial conditions, $a(x, t=0)$, and a boundary condition.  The solution to this is easy to write down:

$$a(x, t) = a(x - ut)$$

any initial profile $a(\xi)$ is simply advected to the right (for $u > 0$) at a velocity $u$.

## Method of solution

To solve this equation, we will need two key peices of coding infustructure:

1. We will need a class that controls our grid and holds our data, and controls our periodic boundary conditions
2. We will need a solver that creates the grid object, initializes the data, and advances the solution.

This code will be included in a module `lec19_modules.py`. Before we use this, we will give a brief primer on how to use modules in coding.

# Python Modules

So far, we've been writing our code all in Jupyter.  But when it comes
time to write code that we want to reuse, we want to put it into a
standalone `*.py` file.

Then we can load it on in python (or Jupyter) and use the capabilities
it provides or make it a standalone program that can be run from the
command line.

**Tip:**
Jupyter is great for interactive explorations and sharing your workflow with others
in a self-contained way.  But if there is an operation that you do over and over,
you should put it into a separate module that you import.  That way you only need to
maintain and debug a single instance of the function, and all your workflows can reuse it.


## Editors

There are a number of popular editors for writing python source.  Some
popular ones include:

* spyder: https://www.spyder-ide.org/

* VS Code: https://code.visualstudio.com/

* emacs / vi / nano

emacs / vi / nano are available on the Roar Collab interactive terminal. However, today, we will use a windows-based command-line interface called `gedit`, which has commands to copy and paste things into the hew file like you would expect.


## Standalone module

Here's a very simply module (lets call it `hello.py`):

```python
def hello():
    print("hello")

if __name__ == "__main__":
    hello()
```

There are two ways we can use this.

* Inside of python (or IPython), we can do:

  ```python
  import hello
  hello.hello()
  ```

* From the command line, we can do:

  ```python
  python hello.py &
  ```

Here we see how the `__name__` variable is treated by python:

* If we import our module into python, then `__name__` is set to the module name
* If we run the module from the command line, then `__name__` is set to `__main__`

To create this file, go to your command line in your already-created `lec19` directory, and type the command line
```bash
gedit hello.py &
```
Copy and pasete the code above into the newly-created file, save, and then exit.

To test that the code ran correctly, finish the code below:


In [ ]:
import hello

hello.hello()

## Representing Data

The process of turning a continuous function, $a(x)$ into a finite number of values that can be stored
on a computer is called *discretization*.

We'll divide our domain into a grid and store the function value at each point in our grid:

![finite-difference grid](https://zingale.github.io/computational_astrophysics/_images/fd_grid_basic.png)

and we use the notation $a_i^n = a(x_i,t^n)$ to represent the discrete data in space and time.

## A First Attempt

We might try to build on what we have already learned:

  * a centered difference for $a_x$ is second-order accurate, so that should be desirable for discretizing
    the spatial terms
    
  * we can do a simple explicit Euler method to update the solution in time.

This would look like:

$$\frac{a_i^{n+1} - a_i^n}{\Delta t} = -u \frac{a_{i+1}^n - a_{i-1}^n}{2\Delta x}$$

This is an **explicit update** (the righthand side depends only on the information at the old time&mdash;stuff we know).  As this is first order in time and second order in space, this method is called FTCS (for *forward-time centered space*).

We can rewrite this as:

$$a_i^{n+1} = a_i^n - \frac{C}{2} (a_{i+1}^n - a_{i-1}^n)$$

where we introduce the [Courant number](https://en.wikipedia.org/wiki/Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition), $C$:

$$C = \frac{u\Delta t}{\Delta x}$$

(this is also called the Courant–Friedrichs–Lewy or CFL number).
$C$ is dimensionless and is a measure of what fraction of a zone we cross per timestep. 

Now imagine that our grid has `N` points, numbered `0`, `1`, ... `N-1`.  When we do the update for `a[0]`, the equation above says we need information from zone `a[-1]`&mdash;this is outside our domain.  To accommodate this, and allow us to use the same equation to update all zones, we extend the domain past the boundary, creating *ghost points*:

![finite-difference ghost points](https://zingale.github.io/computational_astrophysics/_images/fd_ghost.png)

The ghost points are used to implement boundary conditions.  For instance, if we are periodic, then that means that $a_0^n = a_{N-1}^n$, so we would set:

$$a_N^n = a_1^n$$
$$a_{-1}^n = a_{N-2}^n$$

## Test problem

To test our solver, we would like a problem with a known solution at any future point in time.  For advection, this is easy, since the advection equation preserves any initial function and just moves it to the right (for $u > 0$) at a velocity $u$.  

Therefore, we can use periodic boundary conditions on a domain $[0, 1]$ and advect for a time $1/u$, one period, and we should get back exactly what we started with.

## Grid class

To start, let's make a class that controls our grid and holds our data, and knows how to fill our periodic boundary conditions.  Python arrays begin at `0`, so to make life easier, we will define `ilo` and `ihi` to
point to the first and last point in the domain (not including ghost points).

![lo hi with finite difference grid](https://zingale.github.io/computational_astrophysics/_images/fd_ghost_lohi.png)

This grid class is contained within the `lec19_module.py` code, in the lectures GitHub repository. Download this module with the following command:
```bash
curl -O https://raw.githubusercontent.com/PsuAstro410/psuastro410.github.io/refs/heads/main/lectures/lec19_module.py
```
Import it with the code below


In [ ]:
import lec19_module as pdemod

This module includes a method to return a scratch array, as well as a plotting module. The important bits are in the module initialization. We include a print-out of the code below:

```python

    def __init__(self, nx, ng=1, xmin=0.0, xmax=1.0):
        """create a grid with nx points, ng ghost points (on each end)
        that runs from [xmin, xmax]"""

        self.xmin = xmin
        self.xmax = xmax
        self.ng = ng
        self.nx = nx

        # python is zero-based.  Make easy integers to know where the
        # real data lives
        self.ilo = ng
        self.ihi = ng+nx-1

        # physical coords
        self.dx = (xmax - xmin)/(nx-1)
        self.x = xmin + (np.arange(nx+2*ng)-ng)*self.dx

        # storage for the solution
        self.a = self.scratch_array()
        self.ainit = self.scratch_array()

```

In addition, we implement periodic boundary conditions using the ghost cells described above, with the following commands:

```python

    def fill_BCs(self):
        """ fill the a single ghostcell with periodic boundary conditions """
        self.a[self.ilo-1] = self.a[self.ihi-1]
        self.a[self.ihi+1] = self.a[self.ilo+1]

```

## FTCS

Our module also includes our solver.  It will create the grid object, initialize the data (by calling a function we pass in) and then advance the solution using FTCS.

We will specific the Courant number, $C$, and use that to get the timestep, $\Delta t$, with the input arguments:
```python
def ftcs_advection(nx, u, C, num_periods=1.0, init_cond=None):
    """solve the linear advection equation using FTCS.  You are required
    to pass in a function f(g), where g is a FDGrid object that sets up
    the initial conditions"""
```
This Courant number is later used in some of the first lines of the function code:
```python
    # time info
    dt = C*g.dx/u
    t = 0.0
    tmax = num_periods*(g.xmax - g.xmin)/np.abs(u)
```
As we will see shortly, the Courant number is an important parameter for numerical methods for advection, so making it the primary input makes sense.

The boundary conditions are re-set in the while loop using
```python
        # fill the boundary conditions
        g.fill_BCs()
```

As is often in these numerical algorithms, most code goes into setting the problem up, with a spare few lines going into the actual algorithm actually solving for the solution. For our code, this is done within the while loop in the commands
```python
        # loop over zones: note since we are periodic and both endpoints
        # are on the computational domain boundary, we don't have to
        # update both g.ilo and g.ihi -- we could set them equal instead.
        # But this is more general
        for i in range(g.ilo, g.ihi+1):
            anew[i] = g.a[i] - 0.5*C*(g.a[i+1] - g.a[i-1])

        # store the updated solution
        g.a[:] = anew[:]
```

Now we are ready to solve for our solution. We need a function to initialize the state.  We'll do a tophat:

In [ ]:
def tophat(g):
    g.a[:] = 0.0
    g.a[np.logical_and(g.x >= 1./3, g.x <= 2./3.)] = 1.0
    return g

Let's try this out

In [ ]:
nx = 64
u = 1.0
C = 0.5

g = pdemod.ftcs_advection(nx, u, C, init_cond=tophat)
fig = g.plot()
plt.show()

That looks really bad!  Remember, if we did well, then the solution after one period should look just like the initial conditions!

We know that our difference approximations should be more accurate for smaller $\Delta t$ and smaller $\Delta x$.  Reducing $\Delta t$ is equivalent to picking a smaller value of $C$.  So let's try that.

In [ ]:
C = 0.1
g = pdemod.ftcs_advection(nx, u, C, init_cond=tophat)
fig = g.plot()

That's still really bad.  Let's try evolving for less time:

In [ ]:
g = pdemod.ftcs_advection(nx, u, C, init_cond=tophat, num_periods=0.1)
fig = g.plot()

That shows that something is happening near the discontinuities.  Maybe we need more resolution?

In [ ]:
nx = 256
g = pdemod.ftcs_advection(nx, u, C, init_cond=tophat, num_periods=0.1)
fig = g.plot()

## Stability

The issue is that the FTCS method is unstable.  There are different ways that we can analyze this.  We'll do a *truncation analysis*.

Let's start with our discrete equation:

$$a_i^{n+1} = a_i^n - \frac{C}{2} (a_{i+1}^n - a_{i-1}^n)$$

consider the Taylor expansions in space and time:

$$a_{i\pm 1}^n = a_i^n \pm a_x \Delta x + \frac{1}{2} a_{xx} \Delta x^2 + \mathcal{O}(\Delta x^3)$$

$$a_i^{n+1} = a_i^n + a_t \Delta t + \frac{1}{2} a_{tt} \Delta t^2 + \mathcal{O}(\Delta t^3)$$

we can replace the $a_{tt}$ time derivatives in the latter expansion with the analytic equation:

$$a_t = -u a_x$$

$$a_{tt} = -u \frac{\partial}{\partial t} \frac{\partial a}{\partial x} = u^2 a_{xx}$$

and we have:

$$a_i^{n+1} = a_i^n + a_t \Delta t + \frac{1}{2} u^2 a_{xx} \Delta t^2 + \mathcal{O}(\Delta t^3)$$

Now substituting these into our original discrete equation:

$$a_i^n + a_t \Delta t + \frac{1}{2} u^2 a_{xx} \Delta t^2 + \mathcal{O}(\Delta t^3) = a_i^n - \frac{C}{2} (2 a_x \Delta x) + \mathcal{O}(\Delta x^3)$$

or simplifying:

$$a_t + u a_x = -\frac{1}{2} u^2 \Delta t a_{xx} + \mathcal{O}(\Delta t^2) + \mathcal{O}(\Delta x^3)$$

The left side looks like our original linear advection equation,
and the truncation error shows that this is first-order accurate in time.

But when we keep some of the terms in our Taylor expansion, we see that our discrete equation more properly represents an advection equation with a term proportional to $a_{xx}$ on the righthand side.  This looks like a diffusion term. 

**Caution:** The action of diffusion is to take a peaked distribution and smear it out over time.  But our diffusion term has a negative coefficient, so it is more like antidiffusion.  This is unphysical.  And there is no value of $\Delta t$ that makes the diffusion positive, therefore this system is always unstable.

# Upwinding

The FTCS discretization was unstable, so let's try a different discretization.  We can approximate the space derivative using a one-sided difference as either:

$$ (a_x)_i \approx \frac{a_i - a_{i-1}}{\Delta x} + \mathcal{O}(\Delta x)$$

$$ (a_x)_i \approx \frac{a_{i+1} - a_i}{\Delta x} + \mathcal{O}(\Delta x)$$

We'll choose the one that is an [*upwinded* difference](https://en.wikipedia.org/wiki/Upwind_scheme) &mdash;this means we want to make use of the points from the direction that information is flowing.  In our case, $u > 0$, so we want to use the points to the left of $i$ in approximating our derivative.

Choosing this difference makes us first-order accurate in space and time.  Our difference equation is:

$$\frac{a_i^{n+1} - a_i^n}{\Delta t} = - u \frac{a_i^n - a_{i-1}^n}{\Delta x}$$

and our update is:

$$a_i^{n+1} = a_i^n - C (a_i^n - a_{i-1}^n)$$

We'll use the same grid class as before. Our upwinding solver is virtually identical to the FTCS solver we just explored. It is included in `lec19_modules` as `upwind_advection`, the only difference in the code is the following lines:
```python
        for i in range(g.ilo, g.ihi+1):
            anew[i] = g.a[i] - C*(g.a[i] - g.a[i-1])
```

Let's run it for a period.

In [ ]:
def tophat(g):
    g.a[:] = 0.0
    g.a[np.logical_and(g.x >= 1./3, g.x <= 2./3.)] = 1.0
    return g

C = 0.5
u = 1.0
nx = 128

g = pdemod.upwind_advection(nx, u, C, init_cond=tophat)
fig = g.plot()    

This is stable, but not very accurate.  The first-order method is very diffusive.

## Smoother initial conditions

What happens if we start out with smoother initial conditions, like a sine wave?

In [ ]:
def sine(g):
    g.a[:] = np.sin(2 * np.pi * g.x)
    return g

In [ ]:
g = pdemod.upwind_advection(nx, u, C, init_cond=sine)
fig = g.plot()

We see that the solution looks better. Comparing the solution using our two initial conditions suggests that a lot of the troubles we have are near discontinuities.

# Stability

If we do the same stability truncation analysis as with FTCS, we have:

$$a_i^{n+1} = a_i^n - C(a_i^n - a_{i-1}^n)$$

inserting

$$a_{i-1}^n = a_i^n - a_x \Delta x + \frac{1}{2} a_{xx} \Delta x^2 + \mathcal{O}(\Delta x^3)$$

and

$$a_i^{n+1} = a_i^n + a_t \Delta t + \frac{1}{2} u^2 a_{xx} \Delta t^2 + \mathcal{O}(\Delta t^3)$$

and grouping terms, we get:

$$a_t + u a_x = \frac{1}{2} u \Delta x (1 - C) a_{xx} + \mathcal{O}(\Delta x^2) + \mathcal{O}(\Delta t^2)$$

Now we see that the leading error term is $\mathcal{O}(\Delta x)$, so
we are first order accurate, and again, it appears as diffusion, but
the diffusion coefficient is positive (physical) so long as:

$$C < 1$$

This is the stability criterion.

In the next lecture, we will go beyond stability as an important consideration when writing an algorithm.

# Exercises

## Exercise 1 - Muddiest Moment (2 pt)

What was the most confusing part of this lecture?

## Exercise 2 - Practice with coding in modules (8 pt)

Below, we practice coding with modules

### Exercise 2(a) - Create new module (2 pt)

Create a module names `mymodule.py`. Save the following code in the module:
```python
def greeting(name):
  print("Hello, " + name)
```

### Exercise 2(b) - Use the new module (2 pt)

Import and use the new module with the following commands:

```python
import mymodule

mymodule.greeting("Jonathan")
```
Write the commands in the markdown cell below.

In [ ]:
## Answer here


### Exercise 2(c) - Module Variables (2 pt)

The module can contain functions, as already described, but also variables of all types (arrays, dictionaries, objects etc):

Save this code in the file mymodule.py:
```python
person1 = {
  "name": "John",
  "age": 36,
  "country": "Norway"
}
```

### Exercise 2(d) - Access variables (2 pt)

Import the module named mymodule, and access the person1 dictionary:

```python
import mymodule

a = mymodule.person1["age"]
print(a)
```

In [ ]:
## Answer here
